# HPC - Practical Lab: Parallel Limits, Bottlenecks, CPU/GPU Architecture, and AI

<a href="https://colab.research.google.com/github/rubenfonnegra/HPC/blob/master/Sem_02/Hardware_Practical_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/HPC/tree/master/Sem_02/Hardware_Practical_Lab.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


### Learning goals

By the end of the notebook, you should be able to:

- measure execution time correctly;
- compare measured parallel speedup with **Amdahl's Law**;
- calculate **speedup** and **parallel efficiency**;
- distinguish **compute-bound** and **memory-bound** workloads;
- observe the effect of **memory locality and cache behavior**;
- compare CPU and GPU execution for matrix multiplication;
- connect matrix multiplication and attention to modern AI hardware;
- explain *why* a workload behaves the way it does.


**Important:** benchmark results depend on the machine, operating system, CPU, GPU, BLAS implementation, system load, and memory hierarchy. Exact numbers will vary. The goal is to interpret the behavior, not reproduce a specific number.

---

> **📘 High Performance Computing (HPC)**
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution



# 0. Environment and benchmarking utilities

If you are using **Google Colab**, GPU is optional for Experiments 1–3. Nevertheless, GPU is required only for Experiments 4–5.

To enable a GPU in Colab:

**Runtime → Change runtime type → Hardware accelerator → GPU**

In [ ]:
import os
import time
import math
import statistics
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("CPU cores visible to Python:", os.cpu_count())

try:
    import torch
    TORCH_AVAILABLE = True
    CUDA_AVAILABLE = torch.cuda.is_available()
    print("PyTorch:", torch.__version__)
    print("CUDA available:", CUDA_AVAILABLE)
    if CUDA_AVAILABLE:
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    TORCH_AVAILABLE = False
    CUDA_AVAILABLE = False
    print("PyTorch is not available:", e)

In [ ]:
def benchmark(func, repeats=5, warmup=1):
    # Return basic wall-clock timing statistics for a callable.
    for _ in range(warmup):
        func()

    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        func()
        times.append(time.perf_counter() - start)

    return {
        "median_s": statistics.median(times),
        "mean_s": statistics.mean(times),
        "min_s": min(times),
        "max_s": max(times),
        "all_s": times,
    }

def benchmark_torch(func, repeats=10, warmup=3, device="cpu"):
    # CUDA operations are asynchronous, so synchronization is required.
    is_cuda = str(device).startswith("cuda")

    for _ in range(warmup):
        func()
        if is_cuda:
            torch.cuda.synchronize()

    times = []
    for _ in range(repeats):
        if is_cuda:
            torch.cuda.synchronize()

        start = time.perf_counter()
        func()

        if is_cuda:
            torch.cuda.synchronize()

        times.append(time.perf_counter() - start)

    return statistics.median(times)

def pretty_seconds(seconds):
    if seconds < 1e-6:
        return f"{seconds * 1e9:.2f} ns"
    if seconds < 1e-3:
        return f"{seconds * 1e6:.2f} µs"
    if seconds < 1:
        return f"{seconds * 1e3:.2f} ms"
    return f"{seconds:.3f} s"

# Experiment 1 - Why didn't 8 workers make it 8× faster?

We begin with a **controlled synthetic workload**. The goal is to isolate the principle behind Amdahl's Law before introducing implementation-specific effects.

A fraction of the workload is sequential and cannot be parallelized. The remaining part is divided among workers.

Amdahl's Law is

$$
S(N)=\frac{1}{s+\frac{1-s}{N}},
$$

where

- $S(N)$ is the speedup with $N$ workers,
- $s$ is the serial fraction,
- $1-s$ is the parallel fraction.

The theoretical limit is

$$
S_{\max}
=
\lim_{N\to\infty}S(N)
=
\frac{1}{s}.
$$

### Quizz

Suppose $s=0.10$.

- What maximum speedup does Amdahl predict?
- Will 8 workers produce an 8× speedup?
- What do you expect to happen to parallel efficiency as worker count increases?

In [ ]:
SERIAL_FRACTION = 0.10
TOTAL_WORK_SECONDS = 1.0

def synthetic_parallel_run(n_workers,
                           serial_fraction=SERIAL_FRACTION,
                           total_work=TOTAL_WORK_SECONDS):
    # Controlled timing model:
    # 1) execute a serial portion;
    # 2) divide the parallel portion evenly among concurrent workers.
    serial_time = total_work * serial_fraction
    parallel_time = total_work * (1 - serial_fraction)

    start = time.perf_counter()

    time.sleep(serial_time)

    per_worker = parallel_time / n_workers

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = [
            executor.submit(time.sleep, per_worker)
            for _ in range(n_workers)
        ]
        for future in futures:
            future.result()

    return time.perf_counter() - start

workers = [1, 2, 4, 8, 16]
measured_times = []

for n in workers:
    trials = [synthetic_parallel_run(n) for _ in range(3)]
    measured_times.append(statistics.median(trials))

T1 = measured_times[0]

rows = []

for n, tp in zip(workers, measured_times):
    measured_speedup = T1 / tp
    efficiency = measured_speedup / n
    amdahl_speedup = 1 / (
        SERIAL_FRACTION + (1 - SERIAL_FRACTION) / n
    )

    rows.append({
        "Workers": n,
        "Measured time (s)": tp,
        "Measured speedup": measured_speedup,
        "Amdahl speedup": amdahl_speedup,
        "Efficiency": efficiency,
    })

amdahl_df = pd.DataFrame(rows)
amdahl_df.round(4)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    amdahl_df["Workers"],
    amdahl_df["Measured speedup"],
    marker="o",
    label="Measured"
)
plt.plot(
    amdahl_df["Workers"],
    amdahl_df["Amdahl speedup"],
    marker="o",
    label="Amdahl prediction"
)
plt.plot(
    amdahl_df["Workers"],
    amdahl_df["Workers"],
    linestyle="--",
    label="Ideal"
)
plt.xlabel("Workers")
plt.ylabel("Speedup")
plt.title("Measured speedup vs. Amdahl's Law")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    amdahl_df["Workers"],
    amdahl_df["Efficiency"] * 100,
    marker="o"
)
plt.xlabel("Workers")
plt.ylabel("Parallel efficiency (%)")
plt.title("Parallel efficiency")
plt.ylim(0, 105)
plt.grid(alpha=0.2)
plt.show()

print(
    "Theoretical maximum speedup:",
    1 / SERIAL_FRACTION
)

> **HPC connection:** This experiment isolates the theoretical limit. Real programs also pay communication, synchronization, scheduling, and data-movement overhead.

## Explore the Amdahl Wall

In [ ]:
processor_counts = np.arange(1, 129)
serial_fractions = [0.10, 0.05, 0.01]

plt.figure(figsize=(8, 5))

for s in serial_fractions:
    speedup = 1 / (s + (1 - s) / processor_counts)
    plt.plot(
        processor_counts,
        speedup,
        label=f"s = {s:.0%}"
    )

plt.plot(
    processor_counts,
    processor_counts,
    linestyle="--",
    label="Ideal"
)

plt.xlabel("Processors")
plt.ylabel("Speedup")
plt.title("The Amdahl Wall")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# Experiment 2 - Compute-bound or memory-bound?

Two programs can process similarly sized arrays while stressing the hardware very differently.

A useful concept is **arithmetic intensity**:

$$
AI
=
\frac{\text{floating-point operations}}
{\text{bytes transferred}}.
$$

A workload with high arithmetic intensity performs substantial computation for each byte moved. A workload with low arithmetic intensity moves a lot of data while doing relatively little arithmetic.

We will compare:

### Workload A — Dense matrix multiplication

$$
C=AB
$$

For dense $n\times n$ matrices, the operation count is approximately

$$
2n^3 \text{ FLOPs}.
$$

### Workload B — Array copy

$$
B \leftarrow A
$$

This performs very little arithmetic but moves substantial data.

### Quizz

Which workload do you expect to be more compute-intensive? Which one should be more sensitive to memory bandwidth?

In [ ]:
matrix_sizes = [256, 512, 1024, 1536]
compute_rows = []

rng = np.random.default_rng(42)

for n in matrix_sizes:
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)

    matmul_result = benchmark(
        lambda: A @ B,
        repeats=5,
        warmup=1
    )

    copy_result = benchmark(
        lambda: A.copy(),
        repeats=5,
        warmup=1
    )

    approx_flops = 2 * (n ** 3)

    # Copy reads A and writes the destination array.
    approx_copy_bytes = 2 * A.nbytes

    gflops = (
        approx_flops
        / matmul_result["median_s"]
        / 1e9
    )

    copy_bandwidth_gbs = (
        approx_copy_bytes
        / copy_result["median_s"]
        / 1e9
    )

    compute_rows.append({
        "n": n,
        "Matrix multiply time (s)": matmul_result["median_s"],
        "Approx. GFLOP/s": gflops,
        "Array copy time (s)": copy_result["median_s"],
        "Approx. copy bandwidth (GB/s)": copy_bandwidth_gbs,
    })

compute_df = pd.DataFrame(compute_rows)
compute_df.round(4)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    compute_df["n"],
    compute_df["Matrix multiply time (s)"],
    marker="o",
    label="Matrix multiplication"
)

plt.plot(
    compute_df["n"],
    compute_df["Array copy time (s)"],
    marker="o",
    label="Array copy"
)
plt.xlabel("Matrix dimension n")
plt.ylabel("Execution time (s)")
plt.title("Compute-heavy vs. memory-heavy operations")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Interpret carefully

Raw execution time alone does not tell us which resource is saturated because these two workloads perform very different amounts of useful work.

Instead, inspect:

- matrix multiplication throughput in **GFLOP/s**;
- array-copy throughput in **GB/s**;
- how each metric changes with problem size.

For matrix multiplication,

$$
AI
\approx
\frac{2n^3}
{\text{bytes moved}}.
$$

Optimized matrix multiplication also reuses blocks of data through the cache hierarchy.

# Experiment 3 - Why does data access order matter?

Modern CPUs have a memory hierarchy:

$$
\text{Registers}
\rightarrow
L1
\rightarrow
L2
\rightarrow
L3
\rightarrow
DRAM.
$$

NumPy arrays are normally stored in **row-major (C) order**. Consecutive values in a row are adjacent in memory.

We will compare repeatedly summing:

- one **row**: contiguous access;
- one **column**: strided access.

Both operations touch the same number of elements.

### Quizz

Which access pattern should be friendlier to the cache and hardware prefetcher?

In [ ]:
# 4096 x 4096 float32 ≈ 64 MiB, large enough to make memory behavior visible
# while remaining practical on Colab and most laptops.
N = 4096

A = np.random.default_rng(123).random(
    (N, N),
    dtype=np.float32
)

print("Array shape:", A.shape)
print(f"Array size: {A.nbytes / 1024**2:.1f} MiB")
print("C-contiguous:", A.flags["C_CONTIGUOUS"])

row_view = A[N // 2, :]
column_view = A[:, N // 2]

print("Row stride (bytes):", row_view.strides)
print("Column stride (bytes):", column_view.strides)

In [ ]:
REPEATS_INSIDE = 1000

def sum_row_many_times():
    total = 0.0
    for _ in range(REPEATS_INSIDE):
        total += row_view.sum()
    return total

def sum_column_many_times():
    total = 0.0
    for _ in range(REPEATS_INSIDE):
        total += column_view.sum()
    return total

row_time = benchmark(
    sum_row_many_times,
    repeats=5,
    warmup=1
)

column_time = benchmark(
    sum_column_many_times,
    repeats=5,
    warmup=1
)

locality_df = pd.DataFrame([
    {
        "Access pattern": "Contiguous row",
        "Median time (s)": row_time["median_s"],
        "Relative time": 1.0,
    },
    {
        "Access pattern": "Strided column",
        "Median time (s)": column_time["median_s"],
        "Relative time": (
            column_time["median_s"]
            / row_time["median_s"]
        ),
    },
])

locality_df.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(
    locality_df["Access pattern"],
    locality_df["Median time (s)"]
)
plt.ylabel("Median execution time (s)")
plt.title("Contiguous vs. strided memory access")
plt.xticks(rotation=10)
plt.grid(axis="y", alpha=0.2)
plt.show()

> The exact size of the difference is hardware-dependent. The important principle is that **memory layout and access pattern are part of algorithm performance**.

# Experiment 4 - When does the GPU win?

GPUs are designed for high-throughput numerical parallelism. Dense matrix multiplication is particularly suitable because many output elements can be computed concurrently.

For

$$
C=AB,
$$

with

$$
A\in\mathbb{R}^{m\times k},
\qquad
B\in\mathbb{R}^{k\times n},
$$

each output element is

$$
C_{ij}
=
\sum_{r=1}^{k}
A_{ir}B_{rj}.
$$

Many $C_{ij}$ values can be computed independently.

We will compare CPU and GPU matrix multiplication using PyTorch.

> **Benchmarking rule:** CUDA operations are asynchronous. Correct timing requires synchronization.

In [ ]:
if not TORCH_AVAILABLE:
    raise RuntimeError(
        "PyTorch is required for this experiment."
    )

device_gpu = "cuda" if CUDA_AVAILABLE else None

print("CPU device: cpu")
print(
    "GPU device:",
    device_gpu if device_gpu else "Not available"
)

In [ ]:
sizes = [256, 512, 1024, 2048]

if CUDA_AVAILABLE:
    sizes.append(4096)

gpu_rows = []

for n in sizes:
    torch.manual_seed(0)

    A_cpu = torch.randn(
        (n, n),
        dtype=torch.float32
    )
    B_cpu = torch.randn(
        (n, n),
        dtype=torch.float32
    )

    cpu_time = benchmark_torch(
        lambda: A_cpu @ B_cpu,
        repeats=3 if n >= 2048 else 5,
        warmup=1,
        device="cpu"
    )

    row = {
        "n": n,
        "CPU time (s)": cpu_time,
        "GPU time (s)": np.nan,
        "GPU speedup": np.nan,
    }

    if CUDA_AVAILABLE:
        A_gpu = A_cpu.to("cuda")
        B_gpu = B_cpu.to("cuda")

        gpu_time = benchmark_torch(
            lambda: A_gpu @ B_gpu,
            repeats=10,
            warmup=3,
            device="cuda"
        )

        row["GPU time (s)"] = gpu_time
        row["GPU speedup"] = cpu_time / gpu_time

        del A_gpu, B_gpu
        torch.cuda.empty_cache()

    gpu_rows.append(row)

gpu_df = pd.DataFrame(gpu_rows)
gpu_df.round(6)

In [ ]:
if CUDA_AVAILABLE:
    plt.figure(figsize=(8, 5))
    plt.plot(
        gpu_df["n"],
        gpu_df["CPU time (s)"],
        marker="o",
        label="CPU"
    )
    plt.plot(
        gpu_df["n"],
        gpu_df["GPU time (s)"],
        marker="o",
        label="GPU"
    )
    plt.yscale("log")
    plt.xlabel("Matrix dimension n")
    plt.ylabel("Execution time (s, log scale)")
    plt.title("CPU vs. GPU matrix multiplication")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        gpu_df["n"],
        gpu_df["GPU speedup"],
        marker="o"
    )
    plt.axhline(1.0, linestyle="--")
    plt.xlabel("Matrix dimension n")
    plt.ylabel("CPU time / GPU time")
    plt.title("GPU speedup")
    plt.grid(alpha=0.2)
    plt.show()
else:
    print(
        "No CUDA GPU detected. "
        "Enable a GPU in Colab to complete the comparison."
    )

## The data-transfer twist

The previous GPU benchmark kept matrices **already on the GPU**. That measures the computational kernel more directly.

Many real workflows first move data:

$$
\text{CPU memory}
\rightarrow
\text{GPU memory}
\rightarrow
\text{computation}.
$$

For small workloads, transfer and launch overhead can be a large fraction of total runtime.

The next experiment compares:

1. matrix multiplication with data resident on the GPU;
2. CPU→GPU transfer **plus** matrix multiplication.

In [ ]:
if CUDA_AVAILABLE:
    transfer_rows = []

    for n in [256, 512, 1024, 2048]:
        A_cpu = torch.randn(
            (n, n),
            dtype=torch.float32
        )
        B_cpu = torch.randn(
            (n, n),
            dtype=torch.float32
        )

        A_gpu = A_cpu.to("cuda")
        B_gpu = B_cpu.to("cuda")

        resident_time = benchmark_torch(
            lambda: A_gpu @ B_gpu,
            repeats=10,
            warmup=3,
            device="cuda"
        )

        def transfer_and_multiply():
            A_temp = A_cpu.to("cuda")
            B_temp = B_cpu.to("cuda")
            C_temp = A_temp @ B_temp
            return C_temp

        end_to_end = benchmark_torch(
            transfer_and_multiply,
            repeats=5,
            warmup=1,
            device="cuda"
        )

        transfer_rows.append({
            "n": n,
            "GPU resident data (s)": resident_time,
            "Transfer + GPU compute (s)": end_to_end,
            "Overhead factor": end_to_end / resident_time,
        })

        del A_gpu, B_gpu
        torch.cuda.empty_cache()

    transfer_df = pd.DataFrame(transfer_rows)
    display(transfer_df.round(6))
else:
    print("Enable a CUDA GPU to run this section.")

# Experiment 5 - What is an AI layer actually doing?

A fully connected / linear neural-network layer can be written as

$$
Y=XW+b.
$$

For a batch with

$$
X\in\mathbb{R}^{32\times768},
$$

and weights

$$
W\in\mathbb{R}^{768\times3072},
$$

the output is

$$
Y\in\mathbb{R}^{32\times3072}.
$$

The dominant operation is

$$
XW.
$$

### Quizz

Why is this computation attractive for GPUs and specialized matrix hardware such as Tensor Cores?

In [ ]:
batch = 32
in_features = 768
out_features = 3072

X_cpu = torch.randn(
    (batch, in_features),
    dtype=torch.float32
)
W_cpu = torch.randn(
    (in_features, out_features),
    dtype=torch.float32
)
b_cpu = torch.randn(
    (out_features,),
    dtype=torch.float32
)

linear_cpu_time = benchmark_torch(
    lambda: X_cpu @ W_cpu + b_cpu,
    repeats=10,
    warmup=3,
    device="cpu"
)

print(
    "CPU linear layer:",
    pretty_seconds(linear_cpu_time)
)

if CUDA_AVAILABLE:
    X_gpu = X_cpu.to("cuda")
    W_gpu = W_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    linear_gpu_time = benchmark_torch(
        lambda: X_gpu @ W_gpu + b_gpu,
        repeats=30,
        warmup=5,
        device="cuda"
    )

    print(
        "GPU linear layer:",
        pretty_seconds(linear_gpu_time)
    )
    print(
        f"Speedup: "
        f"{linear_cpu_time / linear_gpu_time:.2f}x"
    )

## Attention is matrix math too

Scaled dot-product attention is

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V.
$$

The major computational stages include:

1. $QK^T$ — matrix multiplication;
2. scaling by $1/\sqrt{d_k}$;
3. softmax;
4. multiplication by $V$.

The following implementation is intentionally simple. It exposes the operations rather than reproducing every optimization in production transformer libraries.

In [ ]:
def simple_attention(Q, K, V):
    d_k = Q.shape[-1]

    scores = Q @ K.transpose(-2, -1)
    scores = scores / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)

    return weights @ V

seq_len = 512
d_k = 64

Q_cpu = torch.randn(
    (seq_len, d_k),
    dtype=torch.float32
)
K_cpu = torch.randn(
    (seq_len, d_k),
    dtype=torch.float32
)
V_cpu = torch.randn(
    (seq_len, d_k),
    dtype=torch.float32
)

attention_cpu_time = benchmark_torch(
    lambda: simple_attention(
        Q_cpu,
        K_cpu,
        V_cpu
    ),
    repeats=5,
    warmup=2,
    device="cpu"
)

print(
    "CPU attention:",
    pretty_seconds(attention_cpu_time)
)

if CUDA_AVAILABLE:
    Q_gpu = Q_cpu.to("cuda")
    K_gpu = K_cpu.to("cuda")
    V_gpu = V_cpu.to("cuda")

    attention_gpu_time = benchmark_torch(
        lambda: simple_attention(
            Q_gpu,
            K_gpu,
            V_gpu
        ),
        repeats=5,
        warmup=2,
        device="cuda"
    )

    print(
        "GPU attention:",
        pretty_seconds(attention_gpu_time)
    )
    print(
        f"Speedup: "
        f"{attention_cpu_time / attention_gpu_time:.2f}x"
    )

### Reason about attention

For sequence length $L$,

$$
QK^T
$$

produces an $L\times L$ score matrix.

# Reduced precision and specialized matrix hardware

Modern AI accelerators can execute reduced-precision matrix operations very efficiently.

Depending on the GPU generation, formats such as FP16, BF16, and TF32 may use specialized matrix hardware.

This experiment compares FP32 and FP16 matrix multiplication.

> Results are **hardware-dependent**. Do not assume every GPU generation supports the same formats or achieves the same speedup.

In [ ]:
if CUDA_AVAILABLE:
    n = 2048

    A32 = torch.randn(
        (n, n),
        device="cuda",
        dtype=torch.float32
    )
    B32 = torch.randn(
        (n, n),
        device="cuda",
        dtype=torch.float32
    )

    A16 = A32.to(torch.float16)
    B16 = B32.to(torch.float16)

    fp32_time = benchmark_torch(
        lambda: A32 @ B32,
        repeats=20,
        warmup=5,
        device="cuda"
    )

    fp16_time = benchmark_torch(
        lambda: A16 @ B16,
        repeats=20,
        warmup=5,
        device="cuda"
    )

    precision_df = pd.DataFrame([
        {
            "Precision": "FP32",
            "Median time (s)": fp32_time,
        },
        {
            "Precision": "FP16",
            "Median time (s)": fp16_time,
        },
    ])

    display(precision_df.round(6))

    print(
        f"FP32 / FP16 time ratio: "
        f"{fp32_time / fp16_time:.2f}x"
    )
else:
    print("Enable a CUDA GPU to run this extension.")

# Final Challenge - HPC Hardware Detective

For each workload below, make a prediction **before** reading the suggested discussion.

### Workload A

Copy a 2 GB array from one location in memory to another.

### Workload B

Multiply two dense $4000\times4000$ matrices.

### Workload C

Access elements from a very large array in an irregular/random pattern.

### Workload D

Evaluate a large neural-network linear layer:

$$
Y=XW+b.
$$

For every workload, decide:

1. Is it likely to be **compute-bound**, **memory-bound**, or strongly dependent on both?
2. Would adding CPU cores necessarily help?
3. Is a GPU likely to be useful?
4. Which hardware characteristic matters most?

In [ ]:
challenge = pd.DataFrame({
    "Workload": [
        "A. Copy a 2 GB array",
        "B. Dense 4000×4000 matrix multiplication",
        "C. Large irregular/random memory access",
        "D. Large neural-network linear layer",
    ],
    "Your classification": ["", "", "", ""],
    "CPU or GPU?": ["", "", "", ""],
    "Most important hardware feature": ["", 
                                        "",
                                        "",
                                        "",],
    "Likely overhead / bottleneck": ["", 
                                     "",
                                     "",
                                     "",],
})

challenge

---

## Suggested discussion after predictions

### A — Copy a 2 GB array

Likely dominated by **memory bandwidth**. An array copy performs almost no arithmetic. The system mainly has to:

$read 2 GB+write 2 GB$

So performance is determined primarily by how quickly memory can move data. Very little arithmetic is performed per byte transferred:

$$
AI \approx 0.
$$

Additional arithmetic units do not remove a memory-bandwidth bottleneck.

*Would adding CPU cores help?* Not necessarily.
*Is a GPU useful?* Not necessarily.
*Most important hardware characteristic?* Memory bandwidth.

---

### B — Dense matrix multiplication

Usually has **high arithmetic intensity** and excellent opportunities for parallelism and data reuse. We have:

$$C=AB$$

For square matrices of size n, the approximate operation count is:

$$2n^3$$

For $n=4000$:

$$2(4000)^3 = 128×10^9$$

So this multiplication requires roughly $128 GFLOPs$. A **substantial amount of computation**.

*Would adding CPU cores help?* Yes, up to a point. Matrix multiplication can be divided efficiently among CPU cores.

$$
Core 1 → block \ of \ C \\
Core 2 → block \ of \ C \\
Core 3 → block \ of \ C \\
Core 4 → block \ of \ C \\
...
$$

But scaling eventually becomes limited by $S(N)<N$

*Is a GPU useful?* Very much so. This is nearly an ideal GPU workload because thousands of matrix elements can be computed concurrently.

*Most important hardware characteristic?* Compute throughput (measured in FLOP/s).

---

### C — Irregular/random memory access

The processor performs relatively little computation but repeatedly waits for unpredictable data. The problem is not only **bandwidth** but also **memory latency**.

*Would adding CPU cores help?* Not necessarily.
*Is a GPU useful?* No. Irregular access is much less GPU-friendly due to inefficient memory transactions.
*Most important hardware characteristic?* Memory latency, cache hierarchy, cache size, memory bandwidth

---

### D — Neural-network linear layer

A linear layer is:

$$Y=XW+b$$

The dominant operation here is a dense matrix multiplication ($XW$). This exposes extensive parallelism and maps naturally to GPUs and specialized matrix hardware. If matrices are sufficiently large and efficiently reused, $AI$ is high. This usually has a strong **compute-bound** requirements, although in practice it can depend on both compute and memory.

*Would adding CPU cores help?* Yes, but not indefinitely. The matrix multiplication can be parallelized across cores. However:

$$speedup \neq number \ of \ cores$$

because of overhead and shared hardware resources.

*Is a GPU useful?* Yes. Very useful. Large neural-network layers expose huge amounts of data-level parallelism. For example, many output elements ($Yij​$) can be computed simultaneously.

*Most important hardware characteristic?* Tensor Core / matrix throughput, GPU computational throughput, Memory bandwidth, CPU $↔$ GPU transfer

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Computación Aplicada (HPC) courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Parallel Limits, Bottlenecks, CPU/GPU Architecture, and AI: HPC Notebook*. Institución Universitaria Pascual Bravo.